# Phase 1: Ethereum Phishing Detection - Data Exploration & EDA

## Overview
This notebook explores the XBlock-ETH dataset and BigQuery Ethereum data for phishing detection.

## 1. Setup - Install Dependencies and Authenticate GCP

In [ ]:
import subprocess
import sys
packages = ['google-cloud-storage', 'google-cloud-bigquery', 'kagglehub', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'networkx', 'pyarrow']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
print('All packages installed!')

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from datetime import datetime
import json

from google.cloud import storage, bigquery
import pyarrow.parquet as pq
import pyarrow as pa

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 100)

print('Libraries imported successfully!')

In [ ]:
GCP_PROJECT = os.getenv('GCP_PROJECT', 'eth-phishing-detection')
GCS_BUCKET_RAW = os.getenv('GCS_BUCKET_RAW', 'eth-phishing-raw')
GCS_BUCKET_PROCESSED = os.getenv('GCS_BUCKET_PROCESSED', 'eth-phishing-processed')

print(f'GCP Project: {GCP_PROJECT}')
print(f'Raw bucket: gs://{GCS_BUCKET_RAW}/')
print(f'Processed bucket: gs://{GCS_BUCKET_PROCESSED}/')

try:
    storage_client = storage.Client(project=GCP_PROJECT)
    print('✓ GCS client initialized')
except Exception as e:
    print(f'⚠ GCS init failed: {e}')
    storage_client = None

try:
    bq_client = bigquery.Client(project=GCP_PROJECT)
    print('✓ BigQuery client initialized')
except Exception as e:
    print(f'⚠ BQ init failed: {e}')
    bq_client = None

## 2. Data Sources Overview

- **XBlock-ETH (Kaggle)**: Ethereum phishing transaction network
- **BigQuery**: Large-scale historical Ethereum transactions
- **GCS**: Raw and processed data storage

## 3. Load XBlock-ETH Dataset from Kaggle

In [ ]:
import kagglehub

try:
    xblock_path = kagglehub.dataset_download('xblock/ethereum-phishing-transaction-network')
    print(f'✓ Dataset downloaded to: {xblock_path}')
    files = os.listdir(xblock_path)
    print(f'Files: {sorted(files)}')
except Exception as e:
    print(f'Could not download: {e}')
    xblock_path = '../data/sample'
    if os.path.exists(xblock_path):
        print(f'Using sample data from {xblock_path}')

In [ ]:
transactions_xblock = None
nodes_df = None
edges_df = None

for tx_file in ['transaction_list.csv', 'transactions_sample.csv']:
    tx_path = os.path.join(xblock_path, tx_file)
    if os.path.exists(tx_path):
        transactions_xblock = pd.read_csv(tx_path, low_memory=False)
        print(f'✓ Loaded {tx_file}: {transactions_xblock.shape}')
        break

if transactions_xblock is not None:
    print(transactions_xblock.head())

In [ ]:
for node_file in ['MulDiGraph_node.csv', 'addresses_sample.csv']:
    node_path = os.path.join(xblock_path, node_file)
    if os.path.exists(node_path):
        nodes_df = pd.read_csv(node_path, low_memory=False)
        print(f'✓ Loaded {node_file}: {nodes_df.shape}')
        break

if nodes_df is not None:
    print(nodes_df.head())

In [ ]:
for edge_file in ['MulDiGraph_edge.csv', 'edges_sample.csv']:
    edge_path = os.path.join(xblock_path, edge_file)
    if os.path.exists(edge_path):
        edges_df = pd.read_csv(edge_path, low_memory=False)
        print(f'✓ Loaded {edge_file}: {edges_df.shape}')
        break

if edges_df is not None:
    print(edges_df.head())

## 4. Load BigQuery Data (Optional)

In [ ]:
transactions_bq = None

if bq_client is not None:
    try:
        query = 'SELECT from_address, to_address, value, gas, block_timestamp, transaction_hash FROM `bigquery-public-data.crypto_ethereum.transactions` WHERE DATE(block_timestamp) >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY) LIMIT 100000'
        job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
        query_job = bq_client.query(query, job_config=job_config)
        bytes_scanned = query_job.total_bytes_processed
        cost_usd = (bytes_scanned / (10**9)) * 6.25
        print(f'Estimated cost: ${cost_usd:.2f}')
    except Exception as e:
        print(f'Error: {e}')

## 5. EDA: Class Distribution

In [ ]:
if nodes_df is not None:
    label_cols = [col for col in nodes_df.columns if 'label' in col.lower()]
    if label_cols:
        label_col = label_cols[0]
        labels = nodes_df[[label_col]].copy()
        labels.columns = ['label']
        class_counts = labels['label'].value_counts().sort_index()
        print('CLASS DISTRIBUTION:')
        print(class_counts)
        for class_id, count in class_counts.items():
            pct = count / len(labels) * 100
            print(f'  Class {class_id}: {count} ({pct:.1f}%)')

## 6. EDA: Transaction Statistics

In [ ]:
if transactions_xblock is not None:
    print('TRANSACTION STATISTICS:')
    print(f'  Total: {len(transactions_xblock)}')
    numeric_cols = transactions_xblock.select_dtypes(include=[np.number]).columns
    print(f'  Numeric columns: {list(numeric_cols)}')
    print(transactions_xblock[numeric_cols].describe())

## 7. EDA: Graph Statistics

In [ ]:
import networkx as nx

if edges_df is not None:
    print('Building graph...')
    source_col = next((col for col in edges_df.columns if 'from' in col.lower()), edges_df.columns[0])
    target_col = next((col for col in edges_df.columns if 'to' in col.lower()), edges_df.columns[1] if len(edges_df.columns) > 1 else edges_df.columns[0])
    
    G = nx.DiGraph()
    G.add_edges_from(zip(edges_df[source_col], edges_df[target_col]))
    
    print(f'Nodes: {G.number_of_nodes()}')
    print(f'Edges: {G.number_of_edges()}')
    print(f'Density: {nx.density(G):.6f}')

## 8. Save Data to GCS

In [ ]:
if storage_client is not None and transactions_xblock is not None:
    print('Uploading to GCS...')
    try:
        bucket = storage_client.bucket(GCS_BUCKET_RAW)
        
        import io
        table = pa.Table.from_pandas(transactions_xblock)
        pq_buffer = io.BytesIO()
        pq.write_table(table, pq_buffer)
        pq_buffer.seek(0)
        
        blob = bucket.blob('xblock/transactions.parquet')
        blob.upload_from_file(pq_buffer)
        print(f'✓ Uploaded transactions.parquet')
    except Exception as e:
        print(f'Error: {e}')

## 9. Data Quality Report

In [ ]:
print('DATA QUALITY REPORT')
print('='*70)

if transactions_xblock is not None:
    print(f'\\n[TRANSACTIONS]')
    print(f'  Rows: {len(transactions_xblock)}')
    print(f'  Columns: {len(transactions_xblock.columns)}')
    print(f'  Memory: {transactions_xblock.memory_usage(deep=True).sum() / (1024**2):.2f} MB')
    print(f'  Missing: {transactions_xblock.isnull().sum().sum()}')

if nodes_df is not None:
    print(f'\\n[NODES]')
    print(f'  Rows: {len(nodes_df)}')
    print(f'  Columns: {len(nodes_df.columns)}')

print('\\n' + '='*70)

## 10. Summary

✓ Data loaded and explored
✓ EDA completed
✓ Data saved to GCS

Next: Run 02_feature_engineering.ipynb